In [11]:
fixed_entries = [ { "question": "what is the annual fee", 
                    "answer": "The annual fee is Rs 500.", 
                    "keywords": "fee cost price charge", 
                    "category": "billing" }, 
                  { "question": "how to reset password", 
                    "answer": "Go to Settings > Reset Password.", 
                    "keywords": "password reset login", 
                    "category": "account" }, 
                  { "question": "what are your working hours", 
                    "answer": "We are open 9 AM to 5 PM.", 
                    "keywords": "hours timing open time", 
                    "category": "general" }, 
                  { "question": "how can i pay the fee", 
                    "answer": "You can pay via UPI, card, or net banking.", 
                    "keywords": "pay payment upi fee", "category": "billing" } ]

In [12]:
import pandas as pd
roll_no="1024170300"

In [14]:
last_two_digits=[int(d) for d in roll_no[-2:]]
categories=["billing","account","general"]
personalized_entries=[]
d=last_two_digits[0]
personalized_entries.append({ "question": "how can i check my previous fee payments", 
    "answer": "You can check your previous fee payments from the payment history section.", 
    "keywords": "payment history fees transactions", 
    "category": categories[d % 3] })
d = last_two_digits[1] 
personalized_entries.append({ "question": "can i get a receipt after paying the fee", 
    "answer": "Yes, a payment receipt is generated after the fee payment is completed.", 
    "keywords": "receipt payment fee confirmation", 
    "category": categories[d % 3] })
all_entries = fixed_entries + personalized_entries
df = pd.DataFrame(all_entries)
df

,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i check my previous fee payments,You can check your previous fee payments from ...,payment history fees transactions,billing
5,can i get a receipt after paying the fee,"Yes, a payment receipt is generated after the ...",receipt payment fee confirmation,billing


In [21]:
def score_query(query,df):
    query_words=set(query.lower().split())
    results=[]
    for index, row in df.iterrows():
        entry_text=(row["question"])+" "+row["keywords"]
        entry_words=set(entry_text.lower().split())
        matched_words=query_words.intersection(entry_words)
        if len(query_words)>0:
            confidence= len(matched_words)/len(query_words)
        else:
            confidence=0
        if confidence > 0: 
            results.append({ "index": index, 
                             "question": row["question"], 
                             "answer": row["answer"], 
                             "category": row["category"], 
                             "matched_words": ", ".join(matched_words), 
                             "confidence": confidence })
    results.sort(key=lambda x: x["confidence"], reverse=True )
    return results

#To test
query = input("Enter your query: ")
results = score_query(query, df)
print("\nRanked Results:")
if results:
    for result in results:
        print("\nQuestion:", result["question"])
        print("Answer:", result["answer"])
        print("Category:", result["category"])
        print("Matched Words:", result["matched_words"])
        print("Confidence:", round(result["confidence"], 2))
else:
    print("No matching FAQ found.")


Enter your query:  how can i apply



Ranked Results:

Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.
Category: billing
Matched Words: can, i, how
Confidence: 0.75

Question: how can i check my previous fee payments
Answer: You can check your previous fee payments from the payment history section.
Category: billing
Matched Words: can, i, how
Confidence: 0.75

Question: can i get a receipt after paying the fee
Answer: Yes, a payment receipt is generated after the fee payment is completed.
Category: billing
Matched Words: can, i
Confidence: 0.5

Question: how to reset password
Answer: Go to Settings > Reset Password.
Category: account
Matched Words: how
Confidence: 0.25


In [22]:
def same_category(category_name, df):
    return df[ df["category"].str.lower()==category_name.lower()]
result=same_category("billing",df)
result

,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i check my previous fee payments,You can check your previous fee payments from ...,payment history fees transactions,billing
5,can i get a receipt after paying the fee,"Yes, a payment receipt is generated after the ...",receipt payment fee confirmation,billing


In [24]:
idx=0
print(df.loc[idx])
new=input("Enter the keyword")
df.loc[idx,"keywords"]=(df.loc[idx,"keywords"]+" "+new)
print(df.loc[idx])
filename="1024170300_faq_data.csv"
df.to_csv(filename,index=False)
print("Saved as CSV")

question          what is the annual fee
answer         The annual fee is Rs 500.
keywords    fee cost price charge annual
category                         billing
Name: 0, dtype: object


Enter the keyword annual


question                 what is the annual fee
answer                The annual fee is Rs 500.
keywords    fee cost price charge annual annual
category                                billing
Name: 0, dtype: object
Saved as CSV


In [26]:
count=df.groupby("category").size()
count

category
account    1
billing    4
general    1
dtype: int64

In [30]:
def score_query_modified(query, df):
    query_words = set(query.lower().split())
    results = []
    for index, row in df.iterrows():
        entry_text = row["question"] + " " + row["keywords"]
        entry_words = set(entry_text.lower().split())
        matched_words = query_words.intersection(entry_words)
        if len(query_words) > 0:
            confidence = len(matched_words) / len(query_words)
        else:
            confidence = 0
        if confidence > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "matched_words": ", ".join(matched_words),
                "confidence": confidence
            })
    if not results:
        print("No matching FAQ found.")
        return
    results.sort(
        key=lambda x: x["confidence"],
        reverse=True
    )
    highest_score = results[0]["confidence"]
    top_results = [
        result
        for result in results
        if result["confidence"] == highest_score
    ]
    if len(top_results) > 1:
        print("\nTIE DETECTED!")
        print("Highest Confidence:", highest_score)
    else:
        print("\nNo tie detected.")
    for result in top_results:
        print("\nQuestion:", result["question"])
        print("Answer:", result["answer"])
        print("Category:", result["category"])
        print("Matched Words:", result["matched_words"])
        print("Confidence:", result["confidence"])
# 
query = input("Enter your query: ")
score_query_modified(query, df)

Enter your query:  fee



TIE DETECTED!
Highest Confidence: 1.0

Question: what is the annual fee
Answer: The annual fee is Rs 500.
Category: billing
Matched Words: fee
Confidence: 1.0

Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.
Category: billing
Matched Words: fee
Confidence: 1.0

Question: how can i check my previous fee payments
Answer: You can check your previous fee payments from the payment history section.
Category: billing
Matched Words: fee
Confidence: 1.0

Question: can i get a receipt after paying the fee
Answer: Yes, a payment receipt is generated after the fee payment is completed.
Category: billing
Matched Words: fee
Confidence: 1.0
